# Spam Message Classification

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `spam`

## 2 · Project Overview

We predict whether a message is **spam** using tabularized text features: message length, capital letters, exclamation marks, links, special characters, urgency/money keywords, and sender frequency.

This demonstrates a **tabular approach** to text classification — extracting numeric features from text rather than using raw text models.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given extracted text statistics from a message (length, capitals, links, keyword flags, etc.), predict whether it is spam.

## 5 · Why This Project Matters

- **Spam filtering** is one of the oldest and most impactful ML applications.
- Tabularized text features demonstrate feature engineering from unstructured data.
- Understanding spam signals teaches pattern recognition.
- Directly applicable to email, SMS, and chat filtering.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 7,000 |
| **Features** | msg_length, num_capitals, num_exclamation, num_links, num_special_chars, has_urgency_words, has_money_words, sender_frequency |
| **Target** | spam (0 = not spam, 1 = spam) |
| **Class balance** | ~70/30 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "spam"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: spam
Save dir: E:\Github\Machine-Learning-Projects\Classification\Spam Message Classification


## 11 · Dataset Download or Loading

Synthetic dataset: 7,000 messages with tabularized text features (length, capitals, special chars, etc.) and spam label.

In [4]:
np.random.seed(SEED)
n = 7000
# Simulate tabularized text features (as if extracted from raw messages)
msg_length = np.random.poisson(80, n).clip(5, 500)
num_capitals = np.random.poisson(3, n)
num_exclamation = np.random.poisson(1, n)
num_links = np.random.poisson(0.3, n)
num_special_chars = np.random.poisson(2, n)
has_urgency_words = np.random.binomial(1, 0.15, n)
has_money_words = np.random.binomial(1, 0.1, n)
sender_frequency = np.random.poisson(5, n).clip(1, 50)

score = (-0.002 * msg_length + 0.15 * num_capitals + 0.3 * num_exclamation
         + 0.5 * num_links + 0.1 * num_special_chars + 0.6 * has_urgency_words
         + 0.7 * has_money_words - 0.05 * sender_frequency
         + np.random.normal(0, 0.8, n))
spam = (score > np.percentile(score, 70)).astype(int)

df = pd.DataFrame({"msg_length": msg_length, "num_capitals": num_capitals,
                    "num_exclamation": num_exclamation, "num_links": num_links,
                    "num_special_chars": num_special_chars,
                    "has_urgency_words": has_urgency_words,
                    "has_money_words": has_money_words,
                    "sender_frequency": sender_frequency, "spam": spam})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['spam'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (7000, 9)
Class balance:
spam
0    0.7
1    0.3
Name: proportion, dtype: float64


,msg_length,num_capitals,num_exclamation,num_links,num_special_chars,has_urgency_words,has_money_words,sender_frequency,spam
0,77,7,0,0,1,0,0,2,0
1,86,2,1,0,2,0,0,5,0
2,70,4,2,0,0,0,1,5,1
3,83,4,1,0,1,0,0,6,0
4,90,3,2,0,1,0,1,7,0


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (7000, 9)

Missing values:
msg_length           0
num_capitals         0
num_exclamation      0
num_links            0
num_special_chars    0
has_urgency_words    0
has_money_words      0
sender_frequency     0
spam                 0
dtype: int64

Duplicate rows: 257

Dtypes:
msg_length           int32
num_capitals         int32
num_exclamation      int32
num_links            int32
num_special_chars    int32
has_urgency_words    int32
has_money_words      int32
sender_frequency     int32
spam                 int64
dtype: object

Target 'spam' confirmed. Value counts:
spam
0    4900
1    2100
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = df.columns.drop(TARGET).tolist()
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 4][i % 4]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
plt.suptitle("Feature Distributions by Spam Status", fontsize=14)
plt.tight_layout()
plt.show()

corr = df.corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `spam`.

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "salmon"], edgecolor="black")
ax.set_title("Spam Distribution")
ax.set_xticklabels(["Not Spam (0)", "Spam (1)"], rotation=0)
plt.tight_layout()
plt.show()
print(f"Spam rate: {df[TARGET].mean():.1%}")

Spam rate: 30.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (5600, 8), Test: (1400, 8)
Train class distribution:
spam
0    0.7
1    0.3
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: None — all features are pre-extracted numeric text statistics.
- **Scaling**: Not needed for tree models.
- **Class balance**: ~30% spam — moderate imbalance, stratified split used.
- **Note**: This is a tabularized approach — raw text features (TF-IDF, embeddings) would be used in a full NLP pipeline.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["spam_signal_count"] = (X_train["has_urgency_words"] + X_train["has_money_words"]
                                 + (X_train["num_links"] > 0).astype(int))
X_test["spam_signal_count"] = (X_test["has_urgency_words"] + X_test["has_money_words"]
                                + (X_test["num_links"] > 0).astype(int))

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (9): ['msg_length', 'num_capitals', 'num_exclamation', 'num_links', 'num_special_chars', 'has_urgency_words', 'has_money_words', 'sender_frequency', 'spam_signal_count']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.7736
F1       : 0.5331

              precision    recall  f1-score   support

           0       0.79      0.92      0.85       980
           1       0.70      0.43      0.53       420

    accuracy                           0.77      1400
   macro avg       0.74      0.68      0.69      1400
weighted avg       0.76      0.77      0.76      1400



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                            Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                       
NearestCentroid             0.715000           0.701190  0.788303  0.723471   0.742023  0.715000    0.016885
Perceptron                  0.723571           0.688946  0.767839  0.727933   0.734359  0.723571    0.016072
GaussianNB                  0.747143           0.686054  0.777696  0.743557   0.741142  0.747143    0.017780
SGDClassifier               0.754286           0.685714  0.779265  0.748038   0.745241  0.754286    0.039263
LogisticRegression          0.774286           0.676871  0.801146  0.756249   0.763860  0.774286    0.024428
BernoulliNB                 0.740000           0.676871  0.761579  0.736100   0.733478  0.740000    0.017239
LinearDiscriminantAnalysis  0.773571           0.676361  0.801568  0.755633   0.762927  0.7735

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.7621
F1: 0.5124


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.4900  (1.2s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[64]	valid_0's binary_logloss: 0.490204
LightGBM F1: 0.5060  (0.9s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
Logistic Regression    0.7736  0.5331     0.6988  0.4310
FLAML                  0.7621  0.5124     0.6654  0.4167
LightGBM               0.7629  0.5060     0.6746  0.4048
CatBoost               0.7621  0.4900     0.6867  0.3810

Top 3 models: ['Logistic Regression', 'FLAML', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  Logistic Regression:
    Accuracy : 0.7736
    F1       : 0.5331
    Precision: 0.6988
    Recall   : 0.4310

  FLAML:
    Accuracy : 0.7621
    F1       : 0.5124
    Precision: 0.6654
    Recall   : 0.4167

  LightGBM:
    Accuracy : 0.7629
    F1       : 0.5060
    Precision: 0.6746
    Recall   : 0.4048


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: Logistic Regression

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.92      0.85       980
           1       0.70      0.43      0.53       420

    accuracy                           0.77      1400
   macro avg       0.74      0.68      0.69      1400
weighted avg       0.76      0.77      0.76      1400


Total errors: 317 / 1400 (22.6%)

Sample misclassifications:
      msg_length  num_capitals  num_exclamation  num_links  num_special_chars  has_urgency_words  has_money_words  sender_frequency  spam_signal_count  actual  predicted  correct
12            82             1                1          0                  2                  1                0                 4                  1       1          0    False
4585         109             5                1          1                  1                  0                0                 6                  1       1          0    False
891           65  

## 25 · Interpretation and Business Insight

**Key findings:**
- **Money words** and **urgency words** are the strongest spam indicators.
- **Number of links** strongly correlates with spam.
- **Sender frequency** (trusted senders) is protective.
- The composite `spam_signal_count` is highly predictive.

**Business takeaway:** Flag messages with multiple spam signals (links + urgency + money keywords) for review.

## 26 · Limitations

1. Synthetic features — real spam detection uses raw text.
2. No message content or semantic analysis.
3. Missing sender reputation and IP-based features.
4. No temporal patterns (spam campaigns).
5. Tabular approach misses context and nuance.

## 27 · How to Improve This Project

1. Use raw text with TF-IDF or word embeddings.
2. Add sender reputation and domain features.
3. Include header analysis (routing, authentication).
4. Try NLP models (ModernBERT) for content-based filtering.
5. Implement feedback loop for user-reported spam.

## 28 · Production Considerations

- Deploy as a pre-filter before content-based NLP models.
- Combine with sender reputation scoring.
- Low latency required — tabular models are fast.
- Monitor false positive rate carefully (legitimate email in spam).
- Update features as spam tactics evolve.

## 29 · Common Mistakes

1. Relying only on keyword matching.
2. Not considering false positives (legitimate email marked as spam).
3. Using accuracy on imbalanced data.
4. Ignoring adversarial spam (evasion techniques).
5. Not updating the model as spam evolves.

## 30 · Mini Challenge / Exercises

1. Remove `has_money_words` and `has_urgency_words` — how much does F1 drop?
2. Create `link_density = num_links / msg_length * 100` and test.
3. Try different spam thresholds (20%, 30%, 40%) and compare.
4. Plot PR curve and find optimal threshold.
5. Compare this tabular approach vs. a simple TF-IDF + NB baseline.

## 31 · Final Summary / Key Takeaways

1. **Keyword flags** (urgency, money) are the strongest spam signals.
2. **Links and special characters** add strong signal.
3. **Tabularized text features** enable fast tree-based classification.
4. **False positive control** is critical in spam filtering.
5. **Real-world spam** requires combining tabular + NLP approaches.
6. **Feature engineering from text** is a powerful and fast technique.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Spam Message Classification\metrics.json
{
  "CatBoost": {
    "accuracy": 0.7621,
    "f1": 0.49,
    "precision": 0.6867,
    "recall": 0.381
  },
  "LightGBM": {
    "accuracy": 0.7629,
    "f1": 0.506,
    "precision": 0.6746,
    "recall": 0.4048
  },
  "Logistic Regression": {
    "accuracy": 0.7736,
    "f1": 0.5331,
    "precision": 0.6988,
    "recall": 0.431
  },
  "FLAML": {
    "accuracy": 0.7621,
    "f1": 0.5124,
    "precision": 0.6654,
    "recall": 0.4167
  }
}
